# Temporal analysis

Temporal structure is checked exactly: gap detection over distinct
calendar days, interval regularity over the day-to-day gaps, and
sortedness over adjacent values. The pass needs at least 30 distinct days
before it says anything — below that, a "cadence" would be a guess.

## Modal cadence and gaps

The pass sorts and deduplicates the observed days, then measures the gaps
between consecutive distinct days. The modal gap (most frequent) is the
column's assumed cadence; the median gap is reported alongside it. Gap
semantics depend on the cadence:

- **daily cadence** (modal 1): missing *weekdays* are counted, and a pure
  weekend bridge is not a gap. Holidays are not modeled — the finding's
  reference says so explicitly (`weekdays`, source `inferred`).
- **weekly cadence** (modal 7): gaps are counted in weeks once more than
  ten days elapse.
- **other cadences**: a gap is flagged when the elapsed time exceeds
  1.5× the modal interval *and* the modal interval plus two days; the
  count is expressed in modal-interval units and marked approximate.

At most 10 gap events are reported, ordered by missing count. Each
`gaps` finding carries its reference — the cadence or weekday rule that
defines "missing" — because an empty interval is only meaningful against
an expectation.

## Irregular intervals

The coefficient of variation of the day gaps measures how uneven the
observed spacing is. Above 0.5 (with at least ten gaps), the column gets
an `intervals` statement: irregular spacing, with the modal interval, the
median, and the CV as evidence. The interpretation is practical: an
irregular time index breaks naive resampling and lag features, and the
finding says which column not to trust as a clock.

## Sortedness

A positive-claims-only scan over adjacent non-null values (at least 100
required): a column is either strictly increasing, non-decreasing, or
unclaimed. Sortedness findings are `ordering` statements — "row order
mirrors this column" — and they matter because so many tables arrive
pre-sorted by a column nobody documented. The scan never claims a column
is *unsorted*; it just stays quiet.

In [ ]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(51)

# Frame A: a daily snapshot calendar with a two-weekday hole.
base = dt.date(2024, 1, 1)
skip = {dt.date(2024, 2, 8), dt.date(2024, 2, 9)}
business, i = [], 0
while len(business) < 3_000:
    d = base + dt.timedelta(days=i // 10)
    if d not in skip:
        business.append(d)
    i += 1
daily = pl.DataFrame({"row_id": range(len(business)), "snapshot_date": business})

report = onceover.profile(daily)
for f in report.findings:
    if f["category"] in {"gaps", "ordering"}:
        m = f["evidence"][0]["metrics"]
        print(f"[{f['category']:9s}] {f['claim']}")
        if f["category"] == "gaps":
            print(f"            reference={m.get('reference')} cadence_days={m.get('cadence_days')}")

# Frame B: a bursty event log — an uneven clock with silences.
events, day = [], 0
while len(events) < 3_000:
    for _ in range(5):                # five consecutive active days
        day += 1
        events.append(day)
    day += rng.randrange(9, 15)       # then a silence
log_dates = [dt.date(2024, 1, 1) + dt.timedelta(days=d) for d in events]
rng.shuffle(log_dates)                # logs arrive out of order
log = pl.DataFrame({"event_id": range(len(log_dates)), "event_day": log_dates})

log_report = onceover.profile(log)
log_gaps = [f for f in log_report.findings if f["category"] == "gaps"]
print("gap events in the event log:", len(log_gaps))
for f in log_gaps[:2]:
    print(f"[gaps      ] {f['claim']}")
for f in log_report.findings:
    if f["category"] == "intervals":
        print(f"[intervals ] {f['claim']}")

Reading the output: the snapshot calendar is missing two weekdays —
counted against the weekday reference, with the cadence stated — and it
is non-decreasing with row order, so the table can be range-scanned by
date. The event log gets an irregular-intervals statement instead of a
pretended cadence: its gaps run one day within bursts and ten or more
across silences, and the coefficient of variation says so. The same
column also collects gap events for its silences: against the
within-burst daily modal interval, every silence is a flagged absence,
and the pass caps the report at the ten largest — which is why the print
shows ten events and only two of them. No sortedness claim is made for
the log's date column: it was shuffled on arrival, and the scan only
asserts what holds.

## Datetime columns

Datetime columns are analyzed at day granularity: time-of-day patterns,
intraday gaps, and session boundaries are future work
([](../appendix/directions.ipynb)). What the pass does claim — a missing
weekday, an uneven clock, a row order that mirrors a date — it claims
exactly, over every distinct day, with the reference that makes the
claim checkable.